# AI Builders Club - NetworkEvolution | Session 1
## Script 01 : Hello Gemini - your first AI API call

**Goal:** prove that **your** API key works and see how a *program* (not a chat website) talks to an AI model.

| Part | What you learn |
|------|----------------|
| **Part 1** | One question -> one answer (the minimum API call) |
| **Part 2** | Give the AI a **role** + see the **tokens** (what you pay for) |
| **Part 3** | Turn ugly errors into a friendly one-line message |

### How to use this notebook
Every **code cell** below is one section of the script `01_hello_gemini.py`.

1. Create an empty file `01_hello_gemini.py` in your session folder.
2. Copy the code cells **one by one, in order**, and paste each one at the **bottom** of that file.
3. After each paste, run it: `python 01_hello_gemini.py` - and watch the new output appear.

> Markdown cells (like this one) are the explanations - do **not** paste them into the `.py` file.

### Big picture: what is an "AI API call"?
The model does **not** run on your laptop. Your script sends **one HTTPS request** to Google,
Google runs the model, and sends text back.

```
  Your Python script                        Google (Gemini model)
     |  -- request: key + model + your text -->  |
     |                                           |  model "thinks"
     |  <-- response: answer text + tokens ----  |
```

Same idea as a REST API call to a controller or a switch - only the "server" on the other side is an AI model.

### Key words (you should be able to explain these after today)

| Word | Meaning |
|------|---------|
| **LLM** | Large Language Model. Predicts the next piece of text. Gemini, GPT, Claude, Llama are all LLMs. |
| **API key** | Your password for the AI service. Identifies you and counts your usage. Get one at https://aistudio.google.com/apikey |
| **SDK** | Python library that hides the HTTP details. We use `google-genai` -> `from google import genai` |
| **Client** | `genai.Client()` = the object that talks to Google. Reads `GEMINI_API_KEY` from the environment by itself. |
| **Model** | *Which* brain answers. `flash` = fast + cheap (perfect for us), `pro` = smarter, slower, more expensive. |
| **Interactions API** | Google's newest way to talk to Gemini: `client.interactions.create(model=..., input=...)`. One call = one interaction (question + answer). |
| **input** | What you send: your question / your data (the *prompt*). |
| **output_text** | The answer text - a shortcut the SDK gives you. |
| **system_instruction** | The **role + rules** for the model ("You are a senior network engineer..."). |
| **thinking_level** | Gemini 3 models *think* before answering: `minimal` / `low` / `medium` / `high`. More thinking = better on hard problems, but slower + more tokens. |
| **Token** | The unit the model reads and writes (~4 characters, ~3/4 of a word). You pay per token; limits are in tokens. |

---
## Setup (one time - in the terminal, not in the `.py` file)

**1. Install the libraries** (from the session folder):
```bash
pip install -r requirements.txt
```
`requirements.txt` contains: `google-genai`, `netmiko`, `python-dotenv`

**2. Create your `.env` file** with your API key:
```bash
cp .env.example .env
```
Then open `.env` and paste your key:
```
GEMINI_API_KEY=paste-your-key-here
```
Get a free key at **https://aistudio.google.com/apikey** (docs: https://aistudio.google.com/docs/api-key).

> `.env` holds secrets -> **never** share it, **never** commit it to git.

---
## Section 1 : Imports + load the `.env` file

- `load_dotenv()` reads `.env` and puts every line (`KEY=value`) into the **environment** of this program -> no secrets in the code.
- `from google import genai` is the Gemini SDK (`pip install google-genai`).
- `MODEL` - we keep the model name in **one** variable so we can change it in one place.

**Run it:** nothing is printed yet - no error = libraries installed correctly.

> Tip: if you get `ModuleNotFoundError: No module named 'google'` -> run `pip install -r requirements.txt` again
> (and check you are in the right virtual environment).

In [ ]:
# ---------------------------------------------------------------
# SECTION 1 : Imports + load the .env file
# ---------------------------------------------------------------
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()

MODEL = "gemini-3.8-flash"

# Quick check: is the key loaded? (print only the first 4 characters - never the full key!)
key = os.getenv("GEMINI_API_KEY", "")
print("API key loaded:", (key[:4] + "...") if key else "NO - check your .env file")

---
## Section 2 : Create the Gemini client

- **No key in the code!** `genai.Client()` looks for `GEMINI_API_KEY` in the environment (that `load_dotenv()` just filled).
- Creating the client does **not** call Google yet - it only prepares the connection.
- No key found -> `ValueError: No API key was provided`.

> Optional: `genai.Client(api_key="...")` passes the key directly - fine for a quick test, but never commit it.

In [ ]:
# ---------------------------------------------------------------
# SECTION 2 : Create the Gemini client
# ---------------------------------------------------------------
client = genai.Client()
print("Gemini client ready, model:", MODEL)

---
## Section 3 : PART 1 - one question, one answer

The minimum API call needs only two things: **model** + **input**.

- `interaction` = the full response object (answer, id, tokens, ...)
- `interaction.output_text` = only the answer text

**Try it yourself:** change the `input` to a question from *your* daily work, e.g.
*"In 3 short lines: what is VXLAN?"* and run again.

> Reference: https://aistudio.google.com/api-reference/generate-content/generate-content?codelanguage=python

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 : PART 1 - one question, one answer
# ---------------------------------------------------------------
print("=" * 60)
print("PART 1 : one question, one answer")
print("=" * 60)

interaction = client.interactions.create(
    model=MODEL,
    input="In 3 short lines: what is BGP?",
)
print(interaction.output_text)

---
## Section 4 : PART 2 - give the AI a role + see the tokens

| Option | What it does |
|--------|--------------|
| `system_instruction` | **WHO** the model is + the **RULES** it follows (format, length, tone) |
| `generation_config={"thinking_level": "low"}` | **HOW** it answers - `low` thinking = faster + fewer tokens |
| `interaction.usage` | How many **tokens** this call used |

**Token types:**
- **input** - your prompt + system instruction
- **output** - the answer text
- **thinking** - the model's hidden "reasoning" before answering (you pay for these too!)

**Try it yourself:**
1. Change the role to *"You are a CCIE explaining to a manager"* - see how the answer changes.
2. Change `thinking_level` to `"high"` - compare the **thinking** tokens.

> Reference: https://aistudio.google.com/docs/thinking

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 : PART 2 - role (system_instruction) + options + tokens
# ---------------------------------------------------------------
print()
print("=" * 60)
print("PART 2 : role + tokens")
print("=" * 60)

interaction = client.interactions.create(
    model=MODEL,
    input="Explain OSPF vs BGP.",
    system_instruction=(
        "You are a senior network engineer teaching a beginner. "
        "Answer in maximum 4 short bullet points, plain text, no markdown."
    ),
    generation_config={"thinking_level": "low"},
)
print(interaction.output_text)

## Print thoughts

In [ ]:
    for step in interaction.steps:
            if step.type == "thought" and step.summary:
                print("--- Thought Summary ---")
                for content_block in step.summary:
                    if content_block.type == "text":
                        print(content_block.text)
                print("-----------------------\n")

## Print Token Usage

In [ ]:
usage = interaction.usage
print(f"\n[tokens -> input: {usage.total_input_tokens}, "
      f"output: {usage.total_output_tokens}, "
      f"thinking: {usage.total_thought_tokens}, "
      f"total: {usage.total_tokens}]")

---
## Section 5 : PART 3 - friendly errors instead of a 50-line traceback

Real programs must handle failures. Wrap the API call in `try / except`:

- `type(error).__name__` -> **what** went wrong
- `status_code` -> the HTTP status Google sent back

| Error | HTTP | Usual cause |
|-------|------|-------------|
| `ValueError` | - | No API key found (`.env` missing / wrong folder) |
| `BadRequestError` | 400 | Wrong API key, wrong option name |
| `NotFoundError` | 404 | Wrong model name |
| `RateLimitError` | 429 | Free-tier limit - wait a minute |

> The SDK already **retries** 429 / 5xx errors a few times for you.

**Demo:** this section uses a **wrong model name on purpose** so you can see the friendly error.
Then change `BAD_MODEL` to `MODEL` and run again - the error is gone.

In [ ]:
# ---------------------------------------------------------------
# SECTION 5 : PART 3 - friendly errors (try / except)
# ---------------------------------------------------------------
print()
print("=" * 60)
print("PART 3 : friendly errors")
print("=" * 60)

BAD_MODEL = "gemini-does-not-exist"   # wrong on purpose -> try MODEL here too

try:
    interaction = client.interactions.create(
        model=BAD_MODEL,
        input="Say hello to the network engineers.",
    )
    print(interaction.output_text)
except Exception as error:
    print(f"AI call failed -> {type(error).__name__} "
          f"(HTTP {getattr(error, 'status_code', '-')})")
    print(str(error)[:300])

---
## Recap - what you can now explain

- An **AI API call** = one HTTPS request: key + model + input -> answer + tokens.
- The **API key** lives in `.env`, loaded by `load_dotenv()`, picked up by `genai.Client()`.
- `client.interactions.create(model=..., input=...)` is the whole call; `output_text` is the answer.
- `system_instruction` sets the **role and rules**; `thinking_level` trades speed vs depth.
- **Tokens** (input + output + thinking) are what you pay for and what limits are measured in.
- Wrap API calls in `try / except` so users see a clear one-line error.

### Common beginner errors
| Message | Fix |
|---------|-----|
| `ValueError: No API key was provided` | `GEMINI_API_KEY` missing in `.env`, or `.env` is in another folder |
| `BadRequestError` (400) *API key not valid* | Copy the key again from AI Studio |
| `NotFoundError` (404) | Typo in the model name |
| `RateLimitError` (429) | Free-tier rate limit - wait 1 minute |

**Next:** Script 02 - get the running config from a real switch.